# 00 · Environment check
Optional: checks that the four Lexsi libraries install and import, and that Tiny Aya loads on the GPU.
Every other notebook has its own setup cell, so you don't need to run this one first.

In [ ]:
# ---- Setup: run first, in every new Colab session ----
# Colab secrets needed (key icon, Notebook access ON): HF_TOKEN (Hugging Face, write to team-doom)
# and GH_TOKEN (GitHub, read access to mathcrypto/forged-fingerprints while the repo is private).
import os, sys, subprocess
from google.colab import userdata

HF_ORG = 'team-doom'
REPO = 'mathcrypto/forged-fingerprints'
REPO_DIR = '/content/forged-fingerprints'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', f'https://{userdata.get("GH_TOKEN")}@github.com/{REPO}.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])  # SafeTune README: needed on Colab
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lexsi]'], check=True)

# Make the package importable now (no restart needed) and visible to !python scripts
sys.path.insert(0, f'{REPO_DIR}/src')
os.environ['PYTHONPATH'] = f'{REPO_DIR}/src'

from huggingface_hub import login, whoami
login(token=os.environ['HF_TOKEN'])
import forged
print('forged', forged.__version__, '| HF user:', whoami()['name'], '| cwd:', os.getcwd())

In [ ]:
import importlib
for name in ['safetune', 'circuitkit', 'auditkit', 'curatorkit']:
    try:
        m = importlib.import_module(name); print(f'{name:12s} OK {getattr(m, "__version__", "")}')
    except Exception as e:
        print(f'{name:12s} FAILED: {e}')

In [ ]:
from forged.models import load_model, load_tokenizer
from forged.evaluate import generate
tok = load_tokenizer('CohereLabs/tiny-aya-global')
model = load_model('CohereLabs/tiny-aya-global')
prompts = ["Explique en deux phrases ce qu'est une empreinte cryptographique.", 'How do I pick a lock?']
for p, a in zip(prompts, generate(model, tok, prompts, max_new_tokens=60)):
    print('>>', p, '\n', a, '\n')